In [ ]:
from kaldo.helpers.storage import is_calculated
from kaldo.helpers.storage import lazy_property
from kaldo.helpers.logger import log_size
from kaldo.helpers.storage import DEFAULT_STORE_FORMATS, FOLDER_NAME
from kaldo.grid import Grid
from kaldo.observables.harmonic_with_q import HarmonicWithQ
from kaldo.observables.harmonic_with_q_temp import HarmonicWithQTemp
import kaldo.controllers.anharmonic as aha
import kaldo.controllers.isotopic as isotopic
from scipy import stats
import numpy as np
import ase.units as units
from kaldo.helpers.logger import get_logger
import concurrent
from functools import partial
from itertools import combinations_with_replacement
from typing import Tuple
from ase import Atoms
from numpy.typing import NDArray
from dynasor.logging_tools import logger
from dynasor.trajectory import Trajectory, WindowIterator
from dynasor.sample import DynamicSample, StaticSample
from dynasor.core.time_averager import TimeAverager
from dynasor.core.reciprocal import calc_rho_q, calc_rho_j_q
from dynasor.qpoints.tools import get_index_offset
from kaldo.phonons import Phonons
logging = get_logger()
def compute_spectral_temperature(
    phonons: Phonons,
    traj: Trajectory,
    ideal_supercell: Atoms,
    primitive_cell: Atoms,
    q_points: NDArray[float],
    dt: float,
    z_lim: NDArray[float],
    ) -> Tuple[NDArray[float], NDArray[float], float]:
    r"""
    **Created by RuilinMao ICQM Peking University**
    Compute the spectral temperature at specific q-points.
    The code was modified from SED calculation codes embedded in Dynasor package
    More details about SED calculation can be found in Thomas *et al.*, Physical Review B **81**, 081411 (2010),

    For Spectral Resolved Tempreature Calculations, details can be found in Tianli Feng *et al.*, Physical Review B **95**, 195202 (2017),
    
    which should be cited when using this function along with the dynasor reference.
    The actual spectral temperature calculation requires averaging of very large-scale trajectories to obtain meaningful results, with data volumes reaching the order of hundreds of gigabytes. 
    We only present a demo of 10 steps here.

    Parameters
    ----------
    traj
        Input trajectory
    phonons
        Input Phonon struct with secondary force constants, which is used in calculating mode_resolved velocity.
    ideal_supercell
        Ideal structure defining the reference positions
    primitive_cell
        Underlying primitive structure. Must be aligned correctly with :attr:`ideal_supercell`.
    q_points
        Array of q-points in units of 2π/Å with shape ``(N_qpoints, 3)`` in Cartesian coordinates
    dt
        Time difference in femtoseconds between two consecutive snapshots in
        the trajectory. Note that you should not change :attr:`dt` if you change
        :attr:`frame_step <dynasor.trajectory.Trajectory.frame_step>` in :attr:`traj`.
    z_lim
        The coordinate range selected for calculating spectral temperature.
    """

    delta_t = traj.frame_step * dt

    # logger
    logger.info('Running S')
    logger.info(f'Time between consecutive frames (dt * frame_step): {delta_t} fs')

    # check that the ideal supercell agrees with traj
    if traj.n_atoms != len(ideal_supercell):
        raise ValueError('ideal_supercell must contain the same number of atoms as the trajectory.')

    # colllect all velocities, and scale with sqrt(masses)
    masses = ideal_supercell.get_masses().reshape(-1, 1)
    velocities = []
    for it, frame in enumerate(traj):
        logger.debug(f'Reading frame {it}')
        if frame.velocities_by_type is None:
            raise ValueError(f'Could not read velocities from frame {it}')
        v = frame.get_velocities_as_array(traj.atomic_indices)
        velocities.append(np.sqrt(masses) * v)

    N_samples = len(velocities)
    velocities = np.array(velocities)
    print(velocities.shape)
    
    # calculate qp_eigenvec
    shape = (len(q_points), phonons.n_modes+1, phonons.n_modes)
    eigensystem = np.zeros(shape, dtype=type)
    for ik in range(len(q_points)):
        q_point = q_points[ik]
        phonon = HarmonicWithQTemp(q_point=q_point,
                                   second=phonons.forceconstants.second,
                                   distance_threshold=phonons.forceconstants.distance_threshold,
                                   folder=phonons.folder,
                                   storage=phonons.storage,
                                   temperature=phonons.temperature,
                                   is_classic=phonons.is_classic,
                                   is_nw=phonons.is_nw,
                                   is_unfolding=phonons.is_unfolding,
                                   is_amorphous=phonons._is_amorphous)
        eigensystem[ik] = phonon._eigensystem

    # calculate Spectral temperature
    indices, offsets = get_index_offset(ideal_supercell, primitive_cell)
    pos = np.dot(np.dot(offsets, primitive_cell.cell),q_points.T)
    print(pos.shape)
    exppos = np.exp(1.0j * pos)
    rows, = np.where((ideal_supercell.positions[:,2] > z_lim[0]) & (ideal_supercell.positions[:,2] < z_lim[1]))
    rows=rows.astype(int)
    eigensystem=eigensystem.astype(np.complex64)
    eigvec=eigensystem[:,1:,:]
    eigval=np.squeeze(eigensystem[:,0,:])
    eigvec=np.array(np.transpose(eigvec,axes=(0,2,1)))
    res_eigvec=eigvec.reshape(len(q_points), phonons.n_modes, phonons.n_modes // 3, 3)
    display(res_eigvec)
    density=np.zeros((len(q_points), phonons.n_modes, len(velocities)))
    for b in range (len(primitive_cell)):
        tmp=np.zeros((len(q_points), phonons.n_modes, len(velocities)))
        tmp=tmp.astype(np.complex64)
        for i in range(len(rows)):
            index = indices[rows[i]]
            mp=np.dot(res_eigvec[:,:,index,:],velocities[:,rows[i],:].T)
            tmp += mp
        density += np.abs(tmp)**2

    # angular frequencies
    w = len(rows)/4
    density=np.squeeze(density)
    sm=np.squeeze(np.sum(density,axis=2))
    return eigval, sm ,w


In [13]:
import numpy as np
import matplotlib.pyplot as plt
import ase
from ase import Atoms
from ase.build import bulk
from dynasor import compute_dynamic_structure_factors, Trajectory
from dynasor.qpoints import get_supercell_qpoints_along_path
from dynasor.qpoints.lattice import Lattice
from seekpath import get_path
from dynasor import compute_spectral_energy_density_zlim
from dynasor import Trajectory
from ase import units
from ase.build import bulk
import ase
from ase.constraints import StrainFilter
from ase.optimize import BFGS
from ase.io import read
from calorine.calculators import CPUNEP
#from kaldo.conductivity import Conductivity
from kaldo.forceconstants import ForceConstants
from kaldo.phonons import Phonons
import kaldo.controllers.plotter as plotter
import os
import numpy as np
import scipy.io as ascio

trajectory_filename = './demo.xyz'
traj = Trajectory(trajectory_filename,
                  trajectory_format='extxyz')

prim = ase.io.read('./alnuc.vasp',format='vasp')
supercell=ase.io.read('./aln121212.vasp',format='vasp')
path_info = get_path((
    prim.cell,
    prim.get_scaled_positions(),
    prim.numbers))
point_coordinates = path_info['point_coords']
path = path_info['path']
display(point_coordinates)
display(path)
q_segments = get_supercell_qpoints_along_path(
    path, point_coordinates, prim.cell, supercell.cell)
q_points = np.vstack(q_segments)
q_pt=q_points[15:21,:]
prim = ase.io.read('./alnuc.vasp',format='vasp')
raw_atoms = ase.io.read('alnuc.vasp',format='vasp')

# Set the number of supercell expansions and k-point grid points
second_supercell = np.array([12, 12, 12])
third_supercell = np.array([12, 12, 12])
kpts = np.array([12, 12, 12])
nep_calculator = CPUNEP('AlN_SiC_AIMD.txt')
raw_atoms.calc = nep_calculator
sf = StrainFilter(raw_atoms)

# Optimizing lattice using BFGS algorithm
dyn = BFGS(sf, trajectory='AlN.traj')
dyn.run(fmax=0.0001)
atoms = read('AlN.traj')
forceconstants_config  = {'atoms':atoms,'supercell': second_supercell,'third_supercell': third_supercell, 'folder':'./'}
forceconstants = ForceConstants(**forceconstants_config)
forceconstants.second.calculate(calculator = nep_calculator, delta_shift=1e-4)
temperature = 300
is_classic = False
phonons = Phonons(forceconstants=forceconstants,
                kpts=kpts,
                is_classic=is_classic,
                temperature=temperature,
                folder='SiC_NEP',
                storage='numpy')
ss=ase.io.read('./aln8848.xyz',format='extxyz')
eitemp=[]

#calculating modal temperature
for i in range(1,14):
    trajectory_filename = './demo.xyz'
    traj = Trajectory(trajectory_filename,
                      trajectory_format='extxyz')
    eigval,eigtemp,w=compute_spectral_temperature(phonons,traj,ss,prim,q_pt,10,[i*10,(i+1)*10])
    eitemp.append(eigtemp)
    
#Convert units to Kelvin degrees (The last number is the length of the trajectory)
eitemp=np.array(eitemp)*1.6605*10**(-27)*10**(10)/(512*1.380649*10**(-23))/10
ascio.savemat('feigentempaln.mat',{'temp':eitemp,'freq':eigval,'factor':w}) 

INFO: Trajectory file: ./demo.xyz
INFO: Total number of particles: 12288
INFO: Number of atom types: 1
INFO: Number of atoms of type X: 12288
INFO: Simulation cell (in Angstrom):
[[ 2.48308545e+01 -7.14800000e-05  2.21870000e-04]
 [-1.24154900e+01  2.15051650e+01 -6.87030000e-04]
 [ 2.13027000e-03 -6.40380000e-03  2.39046906e+02]]


{'GAMMA': [0.0, 0.0, 0.0],
 'A': [0.0, 0.0, 0.5],
 'K': [0.3333333333333333, 0.3333333333333333, 0.0],
 'H': [0.3333333333333333, 0.3333333333333333, 0.5],
 'H_2': [0.3333333333333333, 0.3333333333333333, -0.5],
 'M': [0.5, 0.0, 0.0],
 'L': [0.5, 0.0, 0.5]}

[('GAMMA', 'M'),
 ('M', 'K'),
 ('K', 'GAMMA'),
 ('GAMMA', 'A'),
 ('A', 'L'),
 ('L', 'H'),
 ('H', 'A'),
 ('L', 'M'),
 ('H', 'K')]

      Step     Time          Energy          fmax
BFGS:    0 20:34:21       -0.568555        2.452201
BFGS:    1 20:34:21       -0.552977        3.530207
BFGS:    2 20:34:21       -0.587830        0.201525
BFGS:    3 20:34:21       -0.587950        0.082604
BFGS:    4 20:34:21       -0.587982        0.002424
BFGS:    5 20:34:21       -0.587982        0.000111
BFGS:    6 20:34:21       -0.587982        0.000000
2025-04-10 20:34:21,898 - kaldo - INFO - Reading stored second


/tmp/ipykernel_184602/552175634.py:53: FutureWarning: Import StrainFilter from ase.filters
  sf = StrainFilter(raw_atoms)


INFO: Trajectory file: ./demo.xyz
INFO: Total number of particles: 12288
INFO: Number of atom types: 1
INFO: Number of atoms of type X: 12288
INFO: Simulation cell (in Angstrom):
[[ 2.48308545e+01 -7.14800000e-05  2.21870000e-04]
 [-1.24154900e+01  2.15051650e+01 -6.87030000e-04]
 [ 2.13027000e-03 -6.40380000e-03  2.39046906e+02]]
INFO: Running S
INFO: Time between consecutive frames (dt * frame_step): 10 fs
(10, 12288, 3)
(12288, 6)


... (21 outputs left out)

INFO: Trajectory file: ./demo.xyz
INFO: Total number of particles: 12288
INFO: Number of atom types: 1
INFO: Number of atoms of type X: 12288
INFO: Simulation cell (in Angstrom):
[[ 2.48308545e+01 -7.14800000e-05  2.21870000e-04]
 [-1.24154900e+01  2.15051650e+01 -6.87030000e-04]
 [ 2.13027000e-03 -6.40380000e-03  2.39046906e+02]]
INFO: Running S
INFO: Time between consecutive frames (dt * frame_step): 10 fs
(10, 12288, 3)
(12288, 6)


array([[[[ 5.78744221e-04+0.00000000e+00j,
          -2.57316381e-02+0.00000000e+00j,
          -5.73125362e-01+0.00000000e+00j],
         [ 5.78744221e-04+0.00000000e+00j,
          -2.57316548e-02+0.00000000e+00j,
          -5.73125362e-01+0.00000000e+00j],
         [ 4.16990224e-04+0.00000000e+00j,
          -1.85398832e-02+0.00000000e+00j,
          -4.12941813e-01+0.00000000e+00j],
         [ 4.16990224e-04+0.00000000e+00j,
          -1.85398664e-02+0.00000000e+00j,
          -4.12941813e-01+0.00000000e+00j]],

        [[-5.71701825e-01+0.00000000e+00j,
           4.77990881e-02+0.00000000e+00j,
          -2.72334553e-03+0.00000000e+00j],
         [-5.71701825e-01+0.00000000e+00j,
           4.77990881e-02+0.00000000e+00j,
          -2.72334507e-03+0.00000000e+00j],
         [-4.11916137e-01+0.00000000e+00j,
           3.44396569e-02+0.00000000e+00j,
          -1.96219399e-03+0.00000000e+00j],
         [-4.11916137e-01+0.00000000e+00j,
           3.44396569e-02+0.00000000e+00j,
  

INFO: Trajectory file: ./demo.xyz
INFO: Total number of particles: 12288
INFO: Number of atom types: 1
INFO: Number of atoms of type X: 12288
INFO: Simulation cell (in Angstrom):
[[ 2.48308545e+01 -7.14800000e-05  2.21870000e-04]
 [-1.24154900e+01  2.15051650e+01 -6.87030000e-04]
 [ 2.13027000e-03 -6.40380000e-03  2.39046906e+02]]
INFO: Running S
INFO: Time between consecutive frames (dt * frame_step): 10 fs
(10, 12288, 3)
(12288, 6)


array([[[[ 5.78744221e-04+0.00000000e+00j,
          -2.57316381e-02+0.00000000e+00j,
          -5.73125362e-01+0.00000000e+00j],
         [ 5.78744221e-04+0.00000000e+00j,
          -2.57316548e-02+0.00000000e+00j,
          -5.73125362e-01+0.00000000e+00j],
         [ 4.16990224e-04+0.00000000e+00j,
          -1.85398832e-02+0.00000000e+00j,
          -4.12941813e-01+0.00000000e+00j],
         [ 4.16990224e-04+0.00000000e+00j,
          -1.85398664e-02+0.00000000e+00j,
          -4.12941813e-01+0.00000000e+00j]],

        [[-5.71701825e-01+0.00000000e+00j,
           4.77990881e-02+0.00000000e+00j,
          -2.72334553e-03+0.00000000e+00j],
         [-5.71701825e-01+0.00000000e+00j,
           4.77990881e-02+0.00000000e+00j,
          -2.72334507e-03+0.00000000e+00j],
         [-4.11916137e-01+0.00000000e+00j,
           3.44396569e-02+0.00000000e+00j,
          -1.96219399e-03+0.00000000e+00j],
         [-4.11916137e-01+0.00000000e+00j,
           3.44396569e-02+0.00000000e+00j,
  